In [41]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path
from IPython.display import display, Markdown

In [42]:
BASE_DIR = Path("..").resolve()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUT_DIR = BASE_DIR / "outputs" / "part2"

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [43]:
df = pd.read_csv(PROCESSED_DIR / "audusd_part1_corrected.csv", parse_dates=["Date"])

print("Shape:", df.shape)
print("Start:", df["Date"].min())
print("End:", df["Date"].max())

df.head()

Shape: (179, 12)
Start: 2009-02-28 00:00:00
End: 2023-12-31 00:00:00


,Date,spot,fx_return,rate_au,rate_us,infl_au,infl_us,actual_fx_change,uip_implied_change,ppp_implied_change,uip_deviation,ppp_deviation
0,2009-02-28,0.642219,0.005827,3.35,0.16,0.000363,0.004961,0.005827,-0.002658,0.004597,-0.008485,-0.001230
1,2009-03-31,0.691802,0.074370,3.25,0.17,0.000363,0.002429,0.074370,-0.002567,0.002065,-0.076937,-0.072304
2,2009-04-30,0.726691,0.049202,3.06,0.04,0.001659,0.002493,0.049202,-0.002517,0.000834,-0.051719,-0.048368
3,2009-05-31,0.801025,0.097391,3.00,0.14,0.001657,0.002885,0.097391,-0.002383,0.001228,-0.099774,-0.096163
4,2009-06-30,0.806192,0.006429,3.00,0.17,0.001654,0.008553,0.006429,-0.002358,0.006899,-0.008787,0.000471


In [44]:
part2 = df[
    (df["Date"] >= "2009-01-01") &
    (df["Date"] <= "2019-12-31")
].dropna(
    subset=[
        "actual_fx_change",
        "uip_implied_change",
        "ppp_implied_change"
    ]
).copy()

print("Part II sample shape:", part2.shape)
print("Start:", part2["Date"].min())
print("End:", part2["Date"].max())

part2[
    [
        "Date",
        "actual_fx_change",
        "uip_implied_change",
        "ppp_implied_change"
    ]
].head()

Part II sample shape: (131, 12)
Start: 2009-02-28 00:00:00
End: 2019-12-31 00:00:00


,Date,actual_fx_change,uip_implied_change,ppp_implied_change
0,2009-02-28,0.005827,-0.002658,0.004597
1,2009-03-31,0.074370,-0.002567,0.002065
2,2009-04-30,0.049202,-0.002517,0.000834
3,2009-05-31,0.097391,-0.002383,0.001228
4,2009-06-30,0.006429,-0.002358,0.006899


In [45]:
y = part2["actual_fx_change"]

X_uip = sm.add_constant(part2["uip_implied_change"])
uip_model = sm.OLS(y, X_uip).fit()

print(uip_model.summary())

                            OLS Regression Results                            
Dep. Variable:       actual_fx_change   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     1.066
Date:                Wed, 13 May 2026   Prob (F-statistic):              0.304
Time:                        21:37:09   Log-Likelihood:                 257.73
No. Observations:                 131   AIC:                            -511.5
Df Residuals:                     129   BIC:                            -505.7
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -0.0031      0

In [46]:
X_ppp = sm.add_constant(part2["ppp_implied_change"])
ppp_model = sm.OLS(y, X_ppp).fit()

print(ppp_model.summary())

                            OLS Regression Results                            
Dep. Variable:       actual_fx_change   R-squared:                       0.004
Model:                            OLS   Adj. R-squared:                 -0.004
Method:                 Least Squares   F-statistic:                    0.4710
Date:                Wed, 13 May 2026   Prob (F-statistic):              0.494
Time:                        21:37:11   Log-Likelihood:                 257.43
No. Observations:                 131   AIC:                            -510.9
Df Residuals:                     129   BIC:                            -505.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                  0.0009      0

In [47]:
table1_regression_specifications = pd.DataFrame({
    "Model": ["UIP", "PPP"],
    "Regression specification": [
        "Δs_t = α + β(iUS,t − iAUS,t) + εt",
        "Δs_t = α + β(πUS,t − πAUS,t) + εt"
    ],
    "Theoretical restrictions": [
        "α = 0, β = 1",
        "α = 0, β = 1"
    ],
    "Null hypothesis": [
        "H0: α = 0 and β = 1",
        "H0: α = 0 and β = 1"
    ],
    "Alternative hypothesis": [
        "H1: α ≠ 0 and/or β ≠ 1",
        "H1: α ≠ 0 and/or β ≠ 1"
    ]
})

display(Markdown("**Table 1. Regression Specifications, Theoretical Restrictions, and Hypotheses**"))
display(table1_regression_specifications)

table1_regression_specifications.to_csv(
    OUT_DIR / "table1_regression_specifications_restrictions_hypotheses.csv",
    index=False
)

**Table 1. Regression Specifications, Theoretical Restrictions, and Hypotheses**

,Model,Regression specification,Theoretical restrictions,Null hypothesis,Alternative hypothesis
0,UIP,"Δs_t = α + β(iUS,t − iAUS,t) + εt","α = 0, β = 1",H0: α = 0 and β = 1,H1: α ≠ 0 and/or β ≠ 1
1,PPP,"Δs_t = α + β(πUS,t − πAUS,t) + εt","α = 0, β = 1",H0: α = 0 and β = 1,H1: α ≠ 0 and/or β ≠ 1


In [48]:
uip_beta_eq_1 = uip_model.t_test("uip_implied_change = 1")
ppp_beta_eq_1 = ppp_model.t_test("ppp_implied_change = 1")

uip_joint = uip_model.f_test("const = 0, uip_implied_change = 1")
ppp_joint = ppp_model.f_test("const = 0, ppp_implied_change = 1")

uip_beta_eq_1_p = float(uip_beta_eq_1.pvalue)
ppp_beta_eq_1_p = float(ppp_beta_eq_1.pvalue)

uip_joint_p = float(uip_joint.pvalue)
ppp_joint_p = float(ppp_joint.pvalue)

In [49]:


table2_regression_results = pd.DataFrame({
    "Model": ["UIP", "PPP"],
    "α": [
        uip_model.params["const"],
        ppp_model.params["const"]
    ],
    "p(α)": [
        uip_model.pvalues["const"],
        ppp_model.pvalues["const"]
    ],
    "β": [
        uip_model.params["uip_implied_change"],
        ppp_model.params["ppp_implied_change"]
    ],
    "p(β=0)": [
        uip_model.pvalues["uip_implied_change"],
        ppp_model.pvalues["ppp_implied_change"]
    ],
    "95% CI β": [
        f"[{uip_model.conf_int().loc['uip_implied_change', 0]:.2f}, {uip_model.conf_int().loc['uip_implied_change', 1]:.2f}]",
        f"[{ppp_model.conf_int().loc['ppp_implied_change', 0]:.2f}, {ppp_model.conf_int().loc['ppp_implied_change', 1]:.2f}]"
    ],
    "p(β=1)": [
        uip_beta_eq_1_p,
        ppp_beta_eq_1_p
    ],
    "Joint p": [
        uip_joint_p,
        ppp_joint_p
    ],
    "R²": [
        uip_model.rsquared,
        ppp_model.rsquared
    ],
    "N": [
        int(uip_model.nobs),
        int(ppp_model.nobs)
    ]
})

numeric_cols = ["α", "p(α)", "β", "p(β=0)", "p(β=1)", "Joint p", "R²"]
table2_regression_results[numeric_cols] = table2_regression_results[numeric_cols].round(4)

display(Markdown("**Table 2. Regression Results for UIP and PPP**"))
display(table2_regression_results)

table2_regression_results.to_csv(
    OUT_DIR / "table2_regression_results_uip_ppp.csv",
    index=False
)

**Table 2. Regression Results for UIP and PPP**

,Model,α,p(α),β,p(β=0),95% CI β,p(β=1),Joint p,R²,N
0,UIP,-0.0031,0.5153,-2.1192,0.3039,"[-6.18, 1.94]",0.1311,0.2262,0.0082,131
1,PPP,0.0009,0.7705,0.6934,0.4938,"[-1.31, 2.69]",0.7620,0.9077,0.0036,131


In [50]:


table2_regression_results = pd.DataFrame({
    "Model": ["UIP", "PPP"],
    "α": [
        uip_model.params["const"],
        ppp_model.params["const"]
    ],
    "p(α)": [
        uip_model.pvalues["const"],
        ppp_model.pvalues["const"]
    ],
    "β": [
        uip_model.params["uip_implied_change"],
        ppp_model.params["ppp_implied_change"]
    ],
    "p(β=0)": [
        uip_model.pvalues["uip_implied_change"],
        ppp_model.pvalues["ppp_implied_change"]
    ],
    "95% CI β": [
        f"[{uip_model.conf_int().loc['uip_implied_change', 0]:.2f}, {uip_model.conf_int().loc['uip_implied_change', 1]:.2f}]",
        f"[{ppp_model.conf_int().loc['ppp_implied_change', 0]:.2f}, {ppp_model.conf_int().loc['ppp_implied_change', 1]:.2f}]"
    ],
    "p(β=1)": [
        uip_beta_eq_1_p,
        ppp_beta_eq_1_p
    ],
    "Joint p": [
        uip_joint_p,
        ppp_joint_p
    ],
    "R²": [
        uip_model.rsquared,
        ppp_model.rsquared
    ],
    "N": [
        int(uip_model.nobs),
        int(ppp_model.nobs)
    ]
})

numeric_cols = ["α", "p(α)", "β", "p(β=0)", "p(β=1)", "Joint p", "R²"]
table2_regression_results[numeric_cols] = table2_regression_results[numeric_cols].round(4)

display(Markdown("**Table 2. Regression Results for UIP and PPP**"))
display(table2_regression_results)

table2_regression_results.to_csv(
    OUT_DIR / "table2_regression_results_uip_ppp.csv",
    index=False
)

**Table 2. Regression Results for UIP and PPP**

,Model,α,p(α),β,p(β=0),95% CI β,p(β=1),Joint p,R²,N
0,UIP,-0.0031,0.5153,-2.1192,0.3039,"[-6.18, 1.94]",0.1311,0.2262,0.0082,131
1,PPP,0.0009,0.7705,0.6934,0.4938,"[-1.31, 2.69]",0.7620,0.9077,0.0036,131


In [51]:


appendix_table_C1_uip = pd.DataFrame({
    "Term": ["α", "β"],
    "Variable": ["Constant", "UIP-implied change"],
    "Coefficient": [
        uip_model.params["const"],
        uip_model.params["uip_implied_change"]
    ],
    "Std. error": [
        uip_model.bse["const"],
        uip_model.bse["uip_implied_change"]
    ],
    "t-stat": [
        uip_model.tvalues["const"],
        uip_model.tvalues["uip_implied_change"]
    ],
    "p-value": [
        uip_model.pvalues["const"],
        uip_model.pvalues["uip_implied_change"]
    ],
    "CI low": [
        uip_model.conf_int().loc["const", 0],
        uip_model.conf_int().loc["uip_implied_change", 0]
    ],
    "CI high": [
        uip_model.conf_int().loc["const", 1],
        uip_model.conf_int().loc["uip_implied_change", 1]
    ]
})

appendix_table_C1_uip = appendix_table_C1_uip.round(4)

display(Markdown("**Appendix Table C1. Full UIP Regression Output**"))
display(appendix_table_C1_uip)

appendix_table_C1_uip.to_csv(
    OUT_DIR / "appendix_table_C1_full_uip_regression_output.csv",
    index=False
)

**Appendix Table C1. Full UIP Regression Output**

,Term,Variable,Coefficient,Std. error,t-stat,p-value,CI low,CI high
0,α,Constant,-0.0031,0.0047,-0.6524,0.5153,-0.0124,0.0063
1,β,UIP-implied change,-2.1192,2.0529,-1.0323,0.3039,-6.1809,1.9424


In [52]:


appendix_table_C2_ppp = pd.DataFrame({
    "Term": ["α", "β"],
    "Variable": ["Constant", "PPP-implied change"],
    "Coefficient": [
        ppp_model.params["const"],
        ppp_model.params["ppp_implied_change"]
    ],
    "Std. error": [
        ppp_model.bse["const"],
        ppp_model.bse["ppp_implied_change"]
    ],
    "t-stat": [
        ppp_model.tvalues["const"],
        ppp_model.tvalues["ppp_implied_change"]
    ],
    "p-value": [
        ppp_model.pvalues["const"],
        ppp_model.pvalues["ppp_implied_change"]
    ],
    "CI low": [
        ppp_model.conf_int().loc["const", 0],
        ppp_model.conf_int().loc["ppp_implied_change", 0]
    ],
    "CI high": [
        ppp_model.conf_int().loc["const", 1],
        ppp_model.conf_int().loc["ppp_implied_change", 1]
    ]
})

appendix_table_C2_ppp = appendix_table_C2_ppp.round(4)

display(Markdown("**Appendix Table C2. Full PPP Regression Output**"))
display(appendix_table_C2_ppp)

appendix_table_C2_ppp.to_csv(
    OUT_DIR / "appendix_table_C2_full_ppp_regression_output.csv",
    index=False
)

**Appendix Table C2. Full PPP Regression Output**

,Term,Variable,Coefficient,Std. error,t-stat,p-value,CI low,CI high
0,α,Constant,0.0009,0.0030,0.2924,0.7705,-0.0051,0.0068
1,β,PPP-implied change,0.6934,1.0104,0.6863,0.4938,-1.3056,2.6924


In [53]:


appendix_table_C3_tests = pd.DataFrame({
    "Model": ["UIP", "UIP", "PPP", "PPP"],
    "Test": [
        "β = 1",
        "α = 0 and β = 1",
        "β = 1",
        "α = 0 and β = 1"
    ],
    "Test type": [
        "t-test",
        "joint F-test",
        "t-test",
        "joint F-test"
    ],
    "Statistic": [
        float(uip_beta_eq_1.tvalue),
        float(uip_joint.fvalue),
        float(ppp_beta_eq_1.tvalue),
        float(ppp_joint.fvalue)
    ],
    "p-value": [
        uip_beta_eq_1_p,
        uip_joint_p,
        ppp_beta_eq_1_p,
        ppp_joint_p
    ],
    "Conclusion": [
        "Do not reject at 5%",
        "Do not reject at 5%",
        "Do not reject at 5%",
        "Do not reject at 5%"
    ]
})

appendix_table_C3_tests[["Statistic", "p-value"]] = appendix_table_C3_tests[["Statistic", "p-value"]].round(4)

display(Markdown("**Appendix Table C3. Restriction Tests**"))
display(appendix_table_C3_tests)

appendix_table_C3_tests.to_csv(
    OUT_DIR / "appendix_table_C3_restriction_tests.csv",
    index=False
)

/var/folders/t4/q542qsks2b35pwzxx6r7rv3r0000gn/T/ipykernel_81351/2333631642.py:16: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  float(uip_beta_eq_1.tvalue),
/var/folders/t4/q542qsks2b35pwzxx6r7rv3r0000gn/T/ipykernel_81351/2333631642.py:18: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  float(ppp_beta_eq_1.tvalue),


**Appendix Table C3. Restriction Tests**

,Model,Test,Test type,Statistic,p-value,Conclusion
0,UIP,β = 1,t-test,-1.5194,0.1311,Do not reject at 5%
1,UIP,α = 0 and β = 1,joint F-test,1.5037,0.2262,Do not reject at 5%
2,PPP,β = 1,t-test,-0.3034,0.7620,Do not reject at 5%
3,PPP,α = 0 and β = 1,joint F-test,0.0969,0.9077,Do not reject at 5%


In [54]:
print("Saved Part II outputs to:", OUT_DIR)

for file in sorted(OUT_DIR.glob("*.csv")):
    print(file.name)

Saved Part II outputs to: /Users/marwanferreira/Desktop/audusd-carry-uip-ppp/outputs/part2
appendix_table_C1_full_uip_regression_output.csv
appendix_table_C2_full_ppp_regression_output.csv
appendix_table_C3_restriction_tests.csv
table1_regression_specifications_restrictions_hypotheses.csv
table2_regression_results_uip_ppp.csv
